# Data labeling & annotation — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def rate(x): return float(np.mean(np.asarray(x)))
def nearest_predict(train_x, train_y, test_x):
    train_x=np.asarray(train_x); train_y=np.asarray(train_y); preds=[]
    for t in np.asarray(test_x):
        preds.append(train_y[int(np.argmin(np.abs(train_x-t)))])
    return np.asarray(preds)
print('setup ok')

## Turning human judgments into measured labels
Annotation is a noisy measurement process. These examples show majority vote, agreement, annotator accuracy, weighted voting, and the effect of label noise on measured accuracy.

In [ ]:
# 1) Majority vote converts multiple annotations into one label
ann=np.array([[1,1,0],[1,0,0],[0,0,0],[1,1,1],[0,1,0]]); votes=ann.mean(axis=1); maj=(votes>=0.5).astype(int)
plt.figure(figsize=(6,3)); plt.bar(range(5),votes); plt.axhline(.5,c='r',ls='--'); plt.title('vote fraction per item')
assert np.array_equal(maj,np.array([1,0,0,1,0]))

In [ ]:
# 2) Cohen kappa discounts agreement that chance would create
a=np.array([1,1,0,1,0]); b=np.array([1,0,0,1,1]); po=(a==b).mean(); pe=a.mean()*b.mean()+(1-a.mean())*(1-b.mean()); k=(po-pe)/(1-pe)
plt.figure(figsize=(6,3)); plt.bar(['observed','chance','kappa'],[po,pe,k]); plt.ylim(0,1); plt.title(f'kappa = {k:.3f}')
assert abs(k-1/6)<1e-12

In [ ]:
# 3) Gold questions estimate annotator reliability
truth=np.array([1,0,0,1,0]); ann=np.array([[1,1,0],[1,0,0],[0,0,0],[1,1,1],[0,1,0]]); acc=(ann==truth[:,None]).mean(axis=0)
plt.figure(figsize=(6,3)); plt.bar(['rater 1','rater 2','rater 3'],acc); plt.ylim(0,1); plt.title('gold-label accuracy by annotator')
assert np.allclose(acc,[0.8,0.8,0.8])

In [ ]:
# 4) Weighted voting trusts more reliable annotators more
ann=np.array([[1,1,0],[1,0,0],[0,0,0],[1,1,1],[0,1,0]]); w=np.array([.8,.6,.8]); score=(ann*w).sum(axis=1)/w.sum(); lab=(score>=.5).astype(int)
plt.figure(figsize=(6,3)); plt.bar(range(5),score); plt.axhline(.5,c='r',ls='--'); plt.title('reliability-weighted vote')
assert np.array_equal(lab,np.array([1,0,0,1,0]))

In [ ]:
# 5) Label noise caps the accuracy you can honestly measure
noise=np.linspace(0,.4,50); apparent=1-noise
plt.figure(figsize=(6,3)); plt.plot(noise,apparent); plt.xlabel('symmetric label noise'); plt.ylabel('max apparent accuracy'); plt.title('noisy labels lower the ceiling')
assert abs(apparent[25]-0.7959183673)<1e-9